# **02 컨텍스트 증강 및 프롬프트 엔지니어링**

### 학습 내용
1. 외부 정보(컨텍스트)를 프롬프트에 주입하기
2. 텍스트 파일을 활용한 정보 제공
3. 프롬프트 템플릿 활용
4. RAG의 기본 원리 이해

## 0. 환경 변수 설정

In [10]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='./.env')

if os.environ.get("OPENAI_API_KEY"):
    print("API Key가 설정되었습니다.")

API Key가 설정되었습니다.


In [11]:
# .env 파일 로드
load_dotenv()

# API 키 확인
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OPENAI_API_KEY가 정상적으로 로드되었습니다.")
    print("API Key ?? ???? ????.")
else:
    print("OPENAI_API_KEY를 찾을 수 없습니다.")
    print(".env 파일을 생성하고 OPENAI_API_KEY를 설정해주세요.")

OPENAI_API_KEY가 정상적으로 로드되었습니다.
API Key: sk-proj-yw...K00A


## 1. LLM 초기화

In [12]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-5.4-mini")

## 2. 외부 정보 없이 질문하기

먼저 LLM이 알지 못하는 특정 정보에 대해 질문해봅시다.

In [13]:
from IPython.display import Markdown, display

# LLM이 모를 가능성이 높은 질문
question = "우리 회사의 여름 휴가 정책이 어떻게 되나요?"

response = llm.invoke(question)
display(Markdown(response.content))

회사마다 여름 휴가 정책이 달라서, **제가 지금은 귀사 내부 규정을 직접 확인할 수는 없습니다.**  
다만 아래 방법으로 확인하시면 가장 정확합니다.

1. **사내 인트라넷 / HR 포털**에서 `복지`, `휴가`, `연차`, `하계휴가` 항목 확인  
2. **인사팀(HR)** 또는 **팀 매니저**에게 문의  
3. **취업규칙 / 복리후생 규정 / 근로계약서**에 여름휴가 관련 조항 확인

보통 회사에서는 이런 식으로 운영됩니다:
- **법정 연차를 자유롭게 사용**
- **별도 하계휴가 2~5일 부여**
- **7~8월 특정 기간에만 사용 가능**
- **부서별 순환 사용 또는 지정 휴가제**
- **연차 대체 없이 유급/무급 여부가 정책에 따라 다름**

원하시면 제가  
- **사내 공지용 문의 메일 문안**을 작성해드리거나,  
- **휴가 정책 확인 체크리스트**를 만들어드릴게요.

LLM은 학습 데이터에 없는 특정 정보(회사 내부 정책, 개인 정보 등)는 답변할 수 없습니다.

이를 해결하기 위해 **외부 정보를 프롬프트에 포함**시킬 수 있습니다.

## 3. 컨텍스트를 직접 추가하여 질문하기

필요한 정보를 프롬프트에 직접 포함시켜 봅시다.

In [14]:
# 회사 정책 정보 (컨텍스트)
context = """
우리 회사 여름 휴가 정책:
- 전 직원은 7월~8월 중 연속 5일의 여름 휴가를 사용할 수 있습니다.
- 휴가 신청은 최소 2주 전에 해야 합니다.
- 부서별로 최소 인원이 유지되어야 하므로 팀장과 사전 협의가 필요합니다.
- 여름 휴가는 연차와 별도로 제공되는 특별 휴가입니다.
"""

# 컨텍스트와 질문을 함께 전달
prompt = f"""
다음 정보를 참고하여 질문에 답변하세요.

[참고 정보]
{context}

[질문]
{question}
"""

response = llm.invoke(prompt)
display(Markdown(response.content))

우리 회사의 여름 휴가 정책은 다음과 같습니다.

- **7월~8월 중 연속 5일**의 여름 휴가를 사용할 수 있습니다.
- **휴가 신청은 최소 2주 전**에 해야 합니다.
- **부서별 최소 인원 유지**를 위해 **팀장과 사전 협의**가 필요합니다.
- 여름 휴가는 **연차와 별도로 제공되는 특별 휴가**입니다.

## 4. 텍스트 파일로부터 정보 읽어오기

실제 상황에서는 정보가 파일, 데이터베이스, 웹 페이지 등에 저장되어 있습니다.

텍스트 파일에서 정보를 읽어와서 프롬프트에 주입해봅시다.

In [15]:
# 먼저 샘플 텍스트 파일을 생성합니다
sample_text = """
상명대학교 AI 서비스 개발 과정 안내

과정명: RAG · AI Agent 기반 실무형 AI 서비스 개발 과정
기간: 2026.08.18 ~ 2026.08.31 (10일, 총 80시간)
장소: 상명대학교 천안캠퍼스

주요 학습 내용:
1. LLM 애플리케이션 개발 기초
2. RAG(검색증강생성) 시스템 구축
3. Text2SQL 기반 데이터 조회 자동화
4. AI Agent 시스템 개발
5. LangGraph를 활용한 워크플로우 구성

최종 포트폴리오:
- RAG · Text2SQL 기반 데이터 조회 시스템
- Tool 기반 AI Agent 시스템
"""

# 파일 저장
with open("course_info.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

print("샘플 텍스트 파일이 생성되었습니다: course_info.txt")

샘플 텍스트 파일이 생성되었습니다: course_info.txt


In [16]:
# 텍스트 파일 읽기
with open("course_info.txt", "r", encoding="utf-8") as f:
    course_context = f.read()

print("파일 내용:")
print(course_context)

파일 내용:

상명대학교 AI 서비스 개발 과정 안내

과정명: RAG · AI Agent 기반 실무형 AI 서비스 개발 과정
기간: 2026.08.18 ~ 2026.08.31 (10일, 총 80시간)
장소: 상명대학교 천안캠퍼스

주요 학습 내용:
1. LLM 애플리케이션 개발 기초
2. RAG(검색증강생성) 시스템 구축
3. Text2SQL 기반 데이터 조회 자동화
4. AI Agent 시스템 개발
5. LangGraph를 활용한 워크플로우 구성

최종 포트폴리오:
- RAG · Text2SQL 기반 데이터 조회 시스템
- Tool 기반 AI Agent 시스템



In [17]:
# 파일에서 읽은 정보를 활용하여 질문하기
question = "이 과정의 학습 기간과 주요 학습 내용을 요약해주세요."

prompt = f"""
다음 과정 안내 정보를 참고하여 질문에 답변하세요.

[과정 정보]
{course_context}

[질문]
{question}
"""

response = llm.invoke(prompt)
display(Markdown(response.content))

이 과정은 **2026.08.18 ~ 2026.08.31** 동안 진행되며, **총 10일 / 80시간**의 실무형 AI 서비스 개발 과정입니다.  
장소는 **상명대학교 천안캠퍼스**입니다.

### 주요 학습 내용 요약
- **LLM 애플리케이션 개발 기초**
- **RAG(검색증강생성) 시스템 구축**
- **Text2SQL 기반 데이터 조회 자동화**
- **AI Agent 시스템 개발**
- **LangGraph를 활용한 워크플로우 구성**

즉, 이 과정은 **RAG와 AI Agent 중심의 실무형 AI 서비스 개발 역량**을 학습하는 데 초점이 맞춰져 있습니다.

## 5. 프롬프트 템플릿 활용하기

LangChain의 `PromptTemplate`을 사용하면 프롬프트를 더 체계적으로 관리할 수 있습니다.

In [18]:
from langchain_core.prompts import PromptTemplate

# 프롬프트 템플릿 정의
template = """
당신은 도움이 되는 AI 어시스턴트입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

# 템플릿에 값 채우기
formatted_prompt = prompt_template.format(
    context=course_context,
    question="이 과정에서 어떤 포트폴리오를 완성하나요?"
)

print("생성된 프롬프트:")
print(formatted_prompt)
print("\n" + "="*80 + "\n")

response = llm.invoke(formatted_prompt)
print("답변:")
display(Markdown(response.content))

생성된 프롬프트:

당신은 도움이 되는 AI 어시스턴트입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.

[참고 정보]

상명대학교 AI 서비스 개발 과정 안내

과정명: RAG · AI Agent 기반 실무형 AI 서비스 개발 과정
기간: 2026.08.18 ~ 2026.08.31 (10일, 총 80시간)
장소: 상명대학교 천안캠퍼스

주요 학습 내용:
1. LLM 애플리케이션 개발 기초
2. RAG(검색증강생성) 시스템 구축
3. Text2SQL 기반 데이터 조회 자동화
4. AI Agent 시스템 개발
5. LangGraph를 활용한 워크플로우 구성

최종 포트폴리오:
- RAG · Text2SQL 기반 데이터 조회 시스템
- Tool 기반 AI Agent 시스템


[질문]
이 과정에서 어떤 포트폴리오를 완성하나요?

[답변]



답변:


이 과정의 최종 포트폴리오는 다음 2가지입니다.

1. **RAG · Text2SQL 기반 데이터 조회 시스템**
2. **Tool 기반 AI Agent 시스템**

즉, 검색증강생성(RAG)과 Text2SQL을 활용한 데이터 조회 서비스, 그리고 도구를 활용하는 AI Agent 시스템을 완성하게 됩니다.

## 6. RAG의 기본 원리 이해

지금까지 실습한 내용이 바로 **RAG(Retrieval-Augmented Generation)** 의 핵심 원리입니다.

### RAG의 기본 흐름

1. **사용자 질문 입력**
2. **관련 문서 검색** (Retrieval)
   - 벡터 데이터베이스, 키워드 검색, 하이브리드 검색 등
3. **검색된 문서를 프롬프트에 주입** (Augmentation)
4. **LLM이 컨텍스트를 바탕으로 답변 생성** (Generation)

현재까지는 문서 검색 없이 직접 컨텍스트를 제공했지만,
다음 실습에서는 **벡터 데이터베이스를 활용한 자동 문서 검색**을 구현합니다.

## 📖 과제 1: 나만의 지식 베이스 만들기

자신이 관심 있는 주제나 전공 분야의 정보를 담은 텍스트 파일을 만들고,
해당 정보를 활용하여 질문-답변 시스템을 구현해보세요.

**예시 주제:**
- 좋아하는 영화/드라마의 줄거리와 등장인물 정보
- 자신의 포트폴리오나 이력서 내용
- 관심 분야의 용어 사전
- 수업 노트나 요약 자료

**구현 요구사항:**
1. 텍스트 파일(.txt) 생성 (최소 200자 이상)
2. 파일 내용을 읽어와서 컨텍스트로 활용
3. 3개 이상의 질문을 만들어 답변 생성

In [19]:
# TODO 1. 나만의 지식 베이스 내용 작성 (200자 이상)
my_knowledge_base = """
[웹 보안 기초 지식]

SQL Injection은 웹 애플리케이션이 사용자의 입력값을 적절하게 검증하지 않고
SQL 쿼리에 직접 포함할 때 발생할 수 있는 취약점이다.
공격자는 입력값을 조작하여 데이터베이스의 정보를 조회하거나 변경하려고 시도할 수 있다.
대표적인 방어 방법으로는 Prepared Statement 또는 Parameterized Query를 사용하는 것이 있다.

XSS(Cross-Site Scripting)는 공격자가 악성 스크립트를 웹 페이지에 삽입하여
다른 사용자의 브라우저에서 실행되도록 만드는 웹 보안 취약점이다.
XSS는 Stored XSS, Reflected XSS, DOM-based XSS 등으로 구분할 수 있다.
방어를 위해서는 사용자 입력을 검증하고, 출력 시 적절한 인코딩을 적용해야 한다.
또한 Content Security Policy(CSP)를 적용하면 XSS 공격의 영향을 줄이는 데 도움이 된다.

CSRF(Cross-Site Request Forgery)는 사용자가 로그인한 상태를 악용하여
사용자의 의도와 관계없이 특정 요청을 서버로 전송하게 만드는 공격이다.
예를 들어 로그인된 사용자가 공격자가 만든 링크를 클릭했을 때
비밀번호 변경이나 정보 수정 요청이 실행될 수 있다.
CSRF Token, SameSite Cookie, 중요 요청에 대한 추가 인증 등을 이용하여 방어할 수 있다.

웹 보안에서는 입력값 검증, 인증과 인가, 세션 관리, 안전한 쿠키 설정,
HTTPS 사용 등이 중요하다. 단순히 공격을 차단하는 것뿐만 아니라
애플리케이션 설계 단계부터 보안을 고려하는 것이 중요하다.
"""

# TODO 2. 파일명 설정
filename = "web_security_knowledge.txt"

# TODO 3. 질문 3개 작성
questions = [
    "SQL Injection은 무엇이며 어떻게 방어할 수 있나요?",
    "XSS에는 어떤 종류가 있고 방어 방법은 무엇인가요?",
    "CSRF 공격은 어떤 방식으로 발생하며 어떻게 예방할 수 있나요?"
]

# 텍스트 파일 생성
with open(filename, "w", encoding="utf-8") as f:
    f.write(my_knowledge_base)

print(f"✓ 파일이 생성되었습니다: {filename}\n")

# 파일 내용 읽기
with open(filename, "r", encoding="utf-8") as f:
    context = f.read()

# 프롬프트 템플릿 정의
from langchain_core.prompts import PromptTemplate

template = """
당신은 정보보안 분야를 설명하는 AI 어시스턴트입니다.
아래의 참고 정보만을 바탕으로 사용자의 질문에 정확하고 이해하기 쉽게 답변하세요.
참고 정보에 없는 내용이라면 임의로 만들어내지 말고,
"제공된 정보에서 확인할 수 없습니다."라고 답변하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

# 각 질문에 대해 답변 생성
from IPython.display import Markdown, display

for i, question in enumerate(questions, 1):
    print(f"\n{'='*80}")
    print(f"질문 {i}: {question}")
    print('='*80)

    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    print("생성된 프롬프트:")
    print(formatted_prompt)
    print("\n" + "="*80 + "\n")

    response = llm.invoke(formatted_prompt)

    print("답변:")
    display(Markdown(response.content))

✓ 파일이 생성되었습니다: web_security_knowledge.txt


질문 1: SQL Injection은 무엇이며 어떻게 방어할 수 있나요?
생성된 프롬프트:

당신은 정보보안 분야를 설명하는 AI 어시스턴트입니다.
아래의 참고 정보만을 바탕으로 사용자의 질문에 정확하고 이해하기 쉽게 답변하세요.
참고 정보에 없는 내용이라면 임의로 만들어내지 말고,
"제공된 정보에서 확인할 수 없습니다."라고 답변하세요.

[참고 정보]

[웹 보안 기초 지식]

SQL Injection은 웹 애플리케이션이 사용자의 입력값을 적절하게 검증하지 않고
SQL 쿼리에 직접 포함할 때 발생할 수 있는 취약점이다.
공격자는 입력값을 조작하여 데이터베이스의 정보를 조회하거나 변경하려고 시도할 수 있다.
대표적인 방어 방법으로는 Prepared Statement 또는 Parameterized Query를 사용하는 것이 있다.

XSS(Cross-Site Scripting)는 공격자가 악성 스크립트를 웹 페이지에 삽입하여
다른 사용자의 브라우저에서 실행되도록 만드는 웹 보안 취약점이다.
XSS는 Stored XSS, Reflected XSS, DOM-based XSS 등으로 구분할 수 있다.
방어를 위해서는 사용자 입력을 검증하고, 출력 시 적절한 인코딩을 적용해야 한다.
또한 Content Security Policy(CSP)를 적용하면 XSS 공격의 영향을 줄이는 데 도움이 된다.

CSRF(Cross-Site Request Forgery)는 사용자가 로그인한 상태를 악용하여
사용자의 의도와 관계없이 특정 요청을 서버로 전송하게 만드는 공격이다.
예를 들어 로그인된 사용자가 공격자가 만든 링크를 클릭했을 때
비밀번호 변경이나 정보 수정 요청이 실행될 수 있다.
CSRF Token, SameSite Cookie, 중요 요청에 대한 추가 인증 등을 이용하여 방어할 수 있다.

웹 보안에서는 입력값 검증, 인증과 인가, 세션 관리, 안전한 쿠키 설정,
HTTPS 사용 등이 중요하다

SQL Injection은 웹 애플리케이션이 사용자의 입력값을 적절히 검증하지 않고 SQL 쿼리에 직접 포함할 때 발생할 수 있는 취약점입니다.  
공격자는 입력값을 조작하여 데이터베이스의 정보를 조회하거나 변경하려고 시도할 수 있습니다.

방어 방법으로는 다음이 있습니다:
- Prepared Statement 또는 Parameterized Query 사용
- 입력값을 적절히 검증하기

즉, 사용자 입력을 SQL 문에 직접 연결하지 않도록 하는 것이 핵심입니다.


질문 2: XSS에는 어떤 종류가 있고 방어 방법은 무엇인가요?
생성된 프롬프트:

당신은 정보보안 분야를 설명하는 AI 어시스턴트입니다.
아래의 참고 정보만을 바탕으로 사용자의 질문에 정확하고 이해하기 쉽게 답변하세요.
참고 정보에 없는 내용이라면 임의로 만들어내지 말고,
"제공된 정보에서 확인할 수 없습니다."라고 답변하세요.

[참고 정보]

[웹 보안 기초 지식]

SQL Injection은 웹 애플리케이션이 사용자의 입력값을 적절하게 검증하지 않고
SQL 쿼리에 직접 포함할 때 발생할 수 있는 취약점이다.
공격자는 입력값을 조작하여 데이터베이스의 정보를 조회하거나 변경하려고 시도할 수 있다.
대표적인 방어 방법으로는 Prepared Statement 또는 Parameterized Query를 사용하는 것이 있다.

XSS(Cross-Site Scripting)는 공격자가 악성 스크립트를 웹 페이지에 삽입하여
다른 사용자의 브라우저에서 실행되도록 만드는 웹 보안 취약점이다.
XSS는 Stored XSS, Reflected XSS, DOM-based XSS 등으로 구분할 수 있다.
방어를 위해서는 사용자 입력을 검증하고, 출력 시 적절한 인코딩을 적용해야 한다.
또한 Content Security Policy(CSP)를 적용하면 XSS 공격의 영향을 줄이는 데 도움이 된다.

CSRF(Cross-Site Request Forgery)는 사용자가 로그인한 상태를 악용하여
사용자의 의도와 관계없이 특정 요청을 서버로 전송하게 만드는 공격이다.
예를 들어 로그인된 사용자가 공격자가 만든 링크를 클릭했을 때
비밀번호 변경이나 정보 수정 요청이 실행될 수 있다.
CSRF Token, SameSite Cookie, 중요 요청에 대한 추가 인증 등을 이용하여 방어할 수 있다.

웹 보안에서는 입력값 검증, 인증과 인가, 세션 관리, 안전한 쿠키 설정,
HTTPS 사용 등이 중요하다. 단순히 공격을 차단하는 것뿐만 아니라
애플리케이션 설계 단계부터 보안을 고려하는 것

XSS(Cross-Site Scripting)는 공격자가 악성 스크립트를 웹 페이지에 삽입하여 다른 사용자의 브라우저에서 실행되도록 만드는 웹 보안 취약점입니다.

### XSS의 종류
참고 정보에 따르면 XSS는 다음과 같이 구분할 수 있습니다.
- Stored XSS
- Reflected XSS
- DOM-based XSS

### 방어 방법
XSS를 방어하기 위해서는 다음이 중요합니다.
- 사용자 입력을 검증하기
- 출력 시 적절한 인코딩 적용하기
- Content Security Policy(CSP) 적용하기

즉, 입력값을 그대로 신뢰하지 말고, 화면에 출력할 때 안전하게 처리하며, CSP로 실행 가능한 스크립트의 범위를 제한하는 것이 도움이 됩니다.


질문 3: CSRF 공격은 어떤 방식으로 발생하며 어떻게 예방할 수 있나요?
생성된 프롬프트:

당신은 정보보안 분야를 설명하는 AI 어시스턴트입니다.
아래의 참고 정보만을 바탕으로 사용자의 질문에 정확하고 이해하기 쉽게 답변하세요.
참고 정보에 없는 내용이라면 임의로 만들어내지 말고,
"제공된 정보에서 확인할 수 없습니다."라고 답변하세요.

[참고 정보]

[웹 보안 기초 지식]

SQL Injection은 웹 애플리케이션이 사용자의 입력값을 적절하게 검증하지 않고
SQL 쿼리에 직접 포함할 때 발생할 수 있는 취약점이다.
공격자는 입력값을 조작하여 데이터베이스의 정보를 조회하거나 변경하려고 시도할 수 있다.
대표적인 방어 방법으로는 Prepared Statement 또는 Parameterized Query를 사용하는 것이 있다.

XSS(Cross-Site Scripting)는 공격자가 악성 스크립트를 웹 페이지에 삽입하여
다른 사용자의 브라우저에서 실행되도록 만드는 웹 보안 취약점이다.
XSS는 Stored XSS, Reflected XSS, DOM-based XSS 등으로 구분할 수 있다.
방어를 위해서는 사용자 입력을 검증하고, 출력 시 적절한 인코딩을 적용해야 한다.
또한 Content Security Policy(CSP)를 적용하면 XSS 공격의 영향을 줄이는 데 도움이 된다.

CSRF(Cross-Site Request Forgery)는 사용자가 로그인한 상태를 악용하여
사용자의 의도와 관계없이 특정 요청을 서버로 전송하게 만드는 공격이다.
예를 들어 로그인된 사용자가 공격자가 만든 링크를 클릭했을 때
비밀번호 변경이나 정보 수정 요청이 실행될 수 있다.
CSRF Token, SameSite Cookie, 중요 요청에 대한 추가 인증 등을 이용하여 방어할 수 있다.

웹 보안에서는 입력값 검증, 인증과 인가, 세션 관리, 안전한 쿠키 설정,
HTTPS 사용 등이 중요하다. 단순히 공격을 차단하는 것뿐만 아니라
애플리케이션 설계 단계부터 보안을

CSRF(Cross-Site Request Forgery)는 **사용자가 로그인한 상태를 악용하여**, 사용자의 의도와 관계없이 **특정 요청이 서버로 전송되게 만드는 공격**입니다.  
예를 들어, 로그인된 사용자가 공격자가 만든 링크를 클릭하면 **비밀번호 변경**이나 **정보 수정** 같은 요청이 실행될 수 있습니다.

예방 방법은 다음과 같습니다.

- **CSRF Token 사용**
- **SameSite Cookie 적용**
- **중요 요청에 대한 추가 인증 적용**

즉, 사용자가 직접 의도한 요청인지 확인할 수 있는 장치를 두어 CSRF 공격을 막습니다.

## 📖 과제 2: 프롬프트 최적화하기

같은 컨텍스트와 질문이라도 프롬프트를 어떻게 구성하느냐에 따라 답변 품질이 달라집니다.

다음 요소들을 추가하여 프롬프트를 개선해보세요:

1. **역할 정의**: "당신은 ~한 전문가입니다"
2. **답변 형식 지정**: "다음 형식으로 답변하세요: ..."
3. **제약 조건**: "정보에 없는 내용은 '정보 없음'이라고 답하세요"
4. **예시 제공**: Few-shot learning (예시 포함)

원본 프롬프트와 개선된 프롬프트의 답변을 비교해보세요.

In [23]:
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display

# 비교할 질문
question = "SQL Injection은 무엇이며 어떻게 방어할 수 있나요?"


# ==========================================
# 1. 원본 프롬프트
# ==========================================

original_template = """
주어진 정보를 바탕으로 질문에 답변하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

original_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=original_template
)


# ==========================================
# 2. 개선된 프롬프트
# ==========================================

improved_template = """
당신은 웹 애플리케이션 보안 분야의 전문가입니다.
보안에 익숙하지 않은 사람도 이해할 수 있도록 쉽고 정확하게 설명하세요.

각 문장에 참고한 [참고 정보]의 실제 원문이 있다면 각 문장 하단에 ">" 로 인용 표시를 하고, 단계별로 설명하세요.

아래의 [참고 정보]만을 사용하여 질문에 답변하세요.
참고 정보에 없는 내용은 추측하거나 임의로 만들어내지 말고
반드시 "정보 없음"이라고 답변하세요.

다음 형식으로 답변하세요.

1. 개념:
   질문에서 다루는 보안 개념을 간단하게 설명합니다.

2. 발생 원인:
   해당 보안 문제가 발생하는 원인을 설명합니다.

3. 위험성:
   어떤 보안 문제가 발생할 수 있는지 설명합니다.

4. 방어 방법:
   참고 정보에 제시된 방어 방법을 정리합니다.

5. 핵심 요약:
   내용을 한 문장으로 정리합니다.


[답변 예시]

질문: XSS란 무엇인가요?

답변:
1. 개념:
XSS는 악성 스크립트가 다른 사용자의 브라우저에서 실행되도록 만드는 웹 보안 취약점입니다.

2. 발생 원인:
사용자 입력을 적절하게 검증하거나 출력 시 인코딩하지 않을 때 발생할 수 있습니다.

3. 위험성:
악성 스크립트가 사용자의 브라우저에서 실행될 수 있습니다.

4. 방어 방법:
입력값 검증, 출력 인코딩, CSP 적용 등의 방법을 사용할 수 있습니다.

5. 핵심 요약:
XSS는 악성 스크립트 실행을 방지하기 위해 입력 검증과 출력 처리가 중요한 취약점입니다.


[참고 정보]
{context}

[질문]
{question}

[답변]
"""

improved_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=improved_template
)


# ==========================================
# 3. 원본 프롬프트 실행
# ==========================================

original_formatted = original_prompt.format(
    context=context,
    question=question
)

original_response = llm.invoke(original_formatted)

print("=" * 80)
print("📌 원본 프롬프트 결과")
print("=" * 80)

display(Markdown(original_response.content))


# ==========================================
# 4. 개선된 프롬프트 실행
# ==========================================

improved_formatted = improved_prompt.format(
    context=context,
    question=question
)

improved_response = llm.invoke(improved_formatted)

print("\n" + "=" * 80)
print("📌 개선된 프롬프트 결과")
print("=" * 80)

display(Markdown(improved_response.content))

📌 원본 프롬프트 결과


SQL Injection은 웹 애플리케이션이 사용자의 입력값을 제대로 검증하지 않고 SQL 쿼리에 직접 넣을 때 발생하는 취약점입니다.  
공격자가 입력값을 조작하면 데이터베이스에서 정보를 조회하거나 수정하는 등 의도하지 않은 동작을 유발할 수 있습니다.

**방어 방법**
- **Prepared Statement / Parameterized Query 사용**: 사용자 입력을 SQL 문과 분리하여 처리합니다.
- **입력값 검증**: 예상한 형식과 범위에 맞는지 확인합니다.
- **최소 권한 원칙 적용**: DB 계정에 필요한 권한만 부여합니다.
- **에러 메시지 노출 최소화**: SQL 오류 정보가 외부에 드러나지 않게 합니다.


📌 개선된 프롬프트 결과


1. 개념:
SQL Injection은 웹 애플리케이션이 사용자의 입력값을 적절하게 검증하지 않고 SQL 쿼리에 직접 포함할 때 발생할 수 있는 취약점입니다.  
> "SQL Injection은 웹 애플리케이션이 사용자의 입력값을 적절하게 검증하지 않고 SQL 쿼리에 직접 포함할 때 발생할 수 있는 취약점이다."

2. 발생 원인:
사용자의 입력값을 검증하지 않은 채 SQL 쿼리에 직접 넣기 때문에 발생합니다.  
> "웹 애플리케이션이 사용자의 입력값을 적절하게 검증하지 않고 SQL 쿼리에 직접 포함할 때 발생할 수 있는 취약점이다."

3. 위험성:
공격자는 입력값을 조작하여 데이터베이스의 정보를 조회하거나 변경하려고 시도할 수 있습니다.  
> "공격자는 입력값을 조작하여 데이터베이스의 정보를 조회하거나 변경하려고 시도할 수 있다."

4. 방어 방법:
대표적인 방어 방법으로는 Prepared Statement 또는 Parameterized Query를 사용하는 것이 있습니다.  
> "대표적인 방어 방법으로는 Prepared Statement 또는 Parameterized Query를 사용하는 것이 있다."

5. 핵심 요약:
SQL Injection은 입력값을 검증하지 않고 SQL 쿼리에 직접 넣을 때 발생하며, Prepared Statement 또는 Parameterized Query로 방어할 수 있습니다.  
> "SQL Injection은 웹 애플리케이션이 사용자의 입력값을 적절하게 검증하지 않고 SQL 쿼리에 직접 포함할 때 발생할 수 있는 취약점이다."
> "대표적인 방어 방법으로는 Prepared Statement 또는 Parameterized Query를 사용하는 것이 있다."

---

### 참고 자료

- [LangChain Prompts 공식 문서](https://python.langchain.com/docs/modules/model_io/prompts/)
- [프롬프트 엔지니어링 가이드](https://www.promptingguide.ai/)